# MCTS, UCT, and PUCT — two 2-player games, one search

**Objective:** watch one search — lazy expansion, UCT/PUCT selection, negamax backprop —
explain itself on two different games, then play against it yourself.

**The story, in order:**
1. **Minimax** — recurse to the end of the game and back up the exact result; exhaustive
   and perfect.
2. **The problem** — branching factor and depth make that exhaustive recursion explode;
   full game-tree search stops scaling long before games get interesting.
3. **The Monte Carlo method** — estimate an unknown value by averaging many random
   samples instead of enumerating every possibility.
4. **MCTS** — apply that idea to a game tree, one simulation at a time, in four phases:
   Selection → Expansion → Simulation/Evaluation → Backpropagation ([Wikipedia: Monte
   Carlo tree search](https://en.wikipedia.org/wiki/Monte_Carlo_tree_search)).
5. **UCT** — treat each move as a multi-armed bandit, balancing exploitation (`Q`)
   against exploration.
6. **The rollout bottleneck** — random playouts are slow and noisy; a learned value
   function replaces them with one evaluation per leaf.
7. **PUCT** — AlphaZero-style selection: exploration weighted by a policy prior instead
   of raw visit counts.
8. **Tree reuse** — reroot onto the move actually played instead of discarding the
   search and starting over.

**What's ahead:**
- The rules and a strategy tip for each game, plus engine self-checks
- Minimax, the Monte Carlo method, and how MCTS combines them
- Pure UCT vs. PUCT with priors, stepped iteration by iteration
- How the exploration constant *c* reshapes the search
- Tree reuse: rerooting instead of restarting
- An opponent ladder and a tiny self-play network
- A configurable search panel, then you vs. MCTS, on both games

**Perfect play, for the record:** Classic tic-tac-toe is a draw with perfect play on both
sides; Numerical (Graham) tic-tac-toe is a first-player (odds) win with perfect play.

**One-line Classic strategy:** center > corner > edge, and always block an immediate
three-in-a-row.

## The two games

**Classic tic-tac-toe.** Player 1 is X, player 2 is O. Turns alternate; first to place
three of their own mark in a row, column, or diagonal wins; a full board with no line is a
draw. *Strategy:* take the center if it's open, corners next, edges last; always block an
opponent's immediate three-in-a-row before doing anything else. With perfect play on both sides, Classic always ends in a draw.

**Numerical (Graham) tic-tac-toe.** Same 3×3 grid, same win-by-line idea, but the marks
are numbers. Player 1 owns the five odd numbers {1,3,5,7,9}; player 2 owns the four even
numbers {2,4,6,8}. Each turn a player places one of their own remaining numbers into any
empty cell. The first line to sum to exactly 15 wins immediately — the three numbers in
that line can belong to either player. *Strategy:* for every open cell, notice which
number from your remaining pool would complete a 15 through it; prefer center and corners
for the same reason as Classic; and watch for lines that mix both players' numbers, since
those are the easiest to miss. With perfect play the first player (odds) wins.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("scripts"))

import random
from dataclasses import replace

import numpy as np
import matplotlib.pyplot as plt

import engine as eng
import mcts as mc
import viz
import scenarios as sc

plt.rcParams["figure.facecolor"] = "white"

## Engine self-checks

In [2]:
eng.self_check()

engine self-check passed
X | X | X
---------------
. | O | .
---------------
O | . | .
 9 |  4 |  2
---------------
 . |  . |  5
---------------
 . |  . |  .


## Why not just search everything?

**Minimax** would settle any position perfectly: recurse to the end of the game assuming
both players always play their best reply, then back the exact result up to the root. On
a 3×3 board that recursion is cheap enough to run outright.

**The problem** is that "cheap enough" doesn't survive contact with a real game. Classic
tic-tac-toe's branching factor (≤9) and depth (≤9) keep exhaustive minimax fast here, but
the same exhaustive recursion on chess (~35 legal moves, ~80-ply games) is astronomically
large — full game-tree search stops scaling long before games get interesting. Something
has to give: sample instead of enumerate.

## The Monte Carlo method, briefly

The [Monte Carlo method](https://en.wikipedia.org/wiki/Monte_Carlo_method) estimates a
quantity that's too expensive to compute exactly by drawing many random samples and
averaging the result — accuracy comes from sample count, not from enumerating every case.

Applied to a game tree, that idea becomes **Monte Carlo Tree Search**: instead of
minimax's exhaustive recursion, spend a fixed budget of simulations, weight that budget
toward the lines that look most promising, and let the averaged outcome stand in for the
exact minimax value. That's the search this notebook builds next, one phase at a time.

## Baseline story: pure UCT vs. PUCT with priors (Classic)

Pure UCT and PUCT explore the identical opening completely differently within the same
10-iteration budget. Each dashboard below steps the first 6 iterations frame by frame;
**Next** keeps going through the remaining recorded frames to a final "search complete"
view, **Prev** steps back, **Reset** always jumps to iteration 1, selection.

**One MCTS iteration, in four steps:**
- **Selection** — from the root, repeatedly pick the highest-scoring action (tried or
  untried) until reaching a node with an untried action or a finished game.
- **Expansion** — if that action was untried, create the one new node for it (lazy
  expansion: the tree never grows past what's actually been explored).
- **Evaluation** — score the new leaf: a random rollout to a finished game, or a
  stand-in/learned value.
- **Backprop** — carry that value back up to the root, **flipping it (`v → 1 − v`) at
  every step**, since the two players alternate and rewards are constant-sum.

These four phases are the standard MCTS loop -- see [Wikipedia's Monte Carlo tree search article](https://en.wikipedia.org/wiki/Monte_Carlo_tree_search) for the general algorithm this notebook specializes.

Selection scores exactly what's in `mcts.py`: UCT uses `Q + c·√(ln(N_parent+1)/N_child)`;
PUCT uses `Q + c·prior·√(N_parent+1)/(1+N_child)`.

**Reading the diagram:** green = this iteration's path · amber = the node just expanded ·
gold ring = current leader among the root's children (highest visits) · faint gray stubs =
untried actions · the root's children are grouped into three labelled bands, **CENTER /
CORNERS / EDGES**.

In [3]:
uct_mcts = sc.build_mcts(sc.SCENARIO_PURE_UCT)
uct_mcts.new_tree(eng.CLASSIC.initial_board())
uct_mcts.run(sc.SCENARIO_PURE_UCT.n_iterations, record=True)
viz.TreeBoardDashboard(eng.CLASSIC, viz.dashboard_frames(uct_mcts, 6),
                        uct_mcts.root().board, title="Pure UCT").display()

In [4]:
best_uct = uct_mcts.best_action("visits")
best_uct_child = uct_mcts.nodes[uct_mcts.root().children[best_uct]]
print(f"After 10 iterations, pure UCT's most-visited root move is "
      f"{best_uct_child.label(eng.CLASSIC)} with N={best_uct_child.N} (the gold ring "
      f"above) -- every legal move got tried at least once first.")

After 10 iterations, pure UCT's most-visited root move is X0 with N=2 (the gold ring above) -- every legal move got tried at least once first.


Random rollouts are the classic bottleneck: every freshly expanded leaf needs a full random playout to a finished game before it yields any signal, and that signal is noisy. AlphaZero-style search removes the rollout entirely, replacing it with one evaluation from a learned value function. Same opening position as above, but now with hand-set priors (center favored) and a stand-in value function instead of a random rollout:

In [5]:
puct_mcts = sc.build_mcts(sc.SCENARIO_PUCT_PRIOR)
puct_mcts.new_tree(eng.CLASSIC.initial_board())
puct_mcts.run(sc.SCENARIO_PUCT_PRIOR.n_iterations, record=True)
viz.TreeBoardDashboard(eng.CLASSIC, viz.dashboard_frames(puct_mcts, 6),
                        puct_mcts.root().board, title="PUCT + priors").display()

In [6]:
best_puct = puct_mcts.best_action("visits")
best_puct_child = puct_mcts.nodes[puct_mcts.root().children[best_puct]]
print(f"After the same 10 iterations, PUCT's most-visited root move is "
      f"{best_puct_child.label(eng.CLASSIC)} with N={best_puct_child.N} -- the prior "
      f"pulled most of the budget toward one line instead of spreading it across all nine.")

After the same 10 iterations, PUCT's most-visited root move is X4 with N=10 -- the prior pulled most of the budget toward one line instead of spreading it across all nine.


Side by side, after all 10 iterations:

In [7]:
fig = viz.render_side_by_side(eng.CLASSIC,
    [("Pure UCT", uct_mcts), ("PUCT + priors", puct_mcts)])
plt.show()

Pure UCT's untried actions score `+inf`, so it always tries every legal move once before it deepens anywhere — a shallow fan after only 10 iterations. PUCT's exploration term is weighted by the prior instead, so it commits its early visits to the center almost immediately and pushes several moves deep down that one line.

## What the exploration constant c changes

Sweeping c holds everything else fixed (PUCT, the same hand-set priors, 10 iterations) and
tracks how concentrated the root's visits end up on its single most-visited move — low c
lets the prior dominate, high c spreads visits across more candidates.

In [8]:
def visit_concentration(cfg):
    m = sc.build_mcts(cfg)
    m.new_tree(eng.CLASSIC.initial_board())
    m.run(cfg.n_iterations, record=False)
    visits = m.visit_distribution()
    total = sum(visits.values())
    return max(visits.values()) / total if total else 0.0

c_values = [0.2, 0.5, 0.8, 1.1, 1.4, 2.0, 3.0]
concentration = [visit_concentration(replace(sc.SCENARIO_PUCT_PRIOR, c=c)) for c in c_values]

viz.plot_parameter_sweep(c_values, concentration, xlabel="exploration constant c",
                          ylabel="share of visits on the best move",
                          title="PUCT visit concentration vs. c")
plt.show()

The dashed line marks where concentration changes fastest across this sweep — the point where the prior stops dominating and exploration starts actively competing with it for visits.

## Tree reuse: keep the subtree, don't start over

Rerooting onto the move MCTS actually played keeps the work it already paid for instead of
discarding it. Below: run 10 iterations, take MCTS's own best move, re-root onto the
resulting position and keep searching for 8 more — versus throwing the tree away and
starting a fresh 8-iteration search from the same position.

In [9]:
reuse_mcts = sc.build_mcts(replace(sc.SCENARIO_PUCT_PRIOR, n_iterations=10, seed=3))
reuse_mcts.new_tree(eng.CLASSIC.initial_board())
reuse_mcts.run(10, record=True)
viz.TreeBoardDashboard(eng.CLASSIC, viz.dashboard_frames(reuse_mcts, 5),
                        reuse_mcts.root().board, title="Before reroot (10 iterations)").display()

In [10]:
best = reuse_mcts.best_action("visits")
reused_visits = reuse_mcts.reroot(best)
reuse_mcts.run(8, record=False)

fresh_mcts = sc.build_mcts(replace(sc.SCENARIO_PUCT_PRIOR, n_iterations=8, seed=4))
fresh_mcts.new_tree(reuse_mcts.root().board)
fresh_mcts.run(8, record=False)

print(f"reroot carried {reused_visits} visits into the new root before the extra 8 "
      f"iterations even ran")
print(f"reused tree ends at N={reuse_mcts.root().N} total root visits; "
      f"fresh tree ends at N={fresh_mcts.root().N}")

fig = viz.render_side_by_side(eng.CLASSIC,
    [("Reused subtree (+8)", reuse_mcts), ("Fresh search (8)", fresh_mcts)])
plt.show()

reroot carried 10 visits into the new root before the extra 8 iterations even ran
reused tree ends at N=18 total root visits; fresh tree ends at N=8


The reused tree's extra 8 iterations build on top of visit counts the first search already paid for; the fresh tree spends all 8 of its iterations rediscovering ground the reused one covered for free.

## Opponent ladder

random → simple heuristic → weak MCTS (4 iterations) → strong MCTS (10 iterations), on
Classic. Each adjacent pair plays 16 games total, split evenly on who moves first, so
first-move advantage doesn't skew the read.

In [11]:
policies = [
    ("random", sc.random_policy),
    ("heuristic", sc.heuristic_policy),
    ("weak MCTS (4 it)", sc.make_mcts_policy(eng.CLASSIC, 4)),
    ("strong MCTS (10 it)", sc.make_mcts_policy(eng.CLASSIC, 10)),
]

rows = []
for i in range(len(policies) - 1):
    name_a, pol_a = policies[i]
    name_b, pol_b = policies[i + 1]
    w_ab = sc.round_robin_eval(eng.CLASSIC, {1: pol_a, 2: pol_b}, n_games=8, seed=10 + i)
    w_ba = sc.round_robin_eval(eng.CLASSIC, {1: pol_b, 2: pol_a}, n_games=8, seed=20 + i)
    rows.append((name_a, name_b, w_ab[1] + w_ba[2], w_ab[2] + w_ba[1], w_ab[0] + w_ba[0]))

header = f"{'A':<20}{'B':<20}{'A wins':>8}{'B wins':>8}{'draws':>8}"
print(header)
print("-" * len(header))
for name_a, name_b, a_wins, b_wins, draws in rows:
    print(f"{name_a:<20}{name_b:<20}{a_wins:>8}{b_wins:>8}{draws:>8}")

A                   B                     A wins  B wins   draws
----------------------------------------------------------------
random              heuristic                  0      13       3
heuristic           weak MCTS (4 it)          16       0       0
weak MCTS (4 it)    strong MCTS (10 it)        3      13       0


## AlphaZero-lite: self-play with a tiny network

This is the rollout-bottleneck fix from the introduction, trained instead of hand-set: a one-hidden-layer numpy network with policy and value heads, trained purely from its own
MCTS-guided self-play — 8 games, 8 MCTS iterations per move, on Classic. Watch the loss
fall as the network's raw guesses get pulled toward MCTS's improved visit distribution.

In [12]:
net, losses = sc.self_play_training_loop(eng.CLASSIC, n_games=8, mcts_iterations=8, seed=5)

plt.figure(figsize=(6, 3.6))
plt.plot(range(1, len(losses) + 1), losses, marker="o", color=viz.COLOR_ROOT)
plt.xlabel("self-play game")
plt.ylabel("avg. loss (policy + value cross-entropy)")
plt.title("AlphaZero-lite training loss", loc="left")
plt.grid(alpha=0.25)
plt.show()

prior, value = net.predict(eng.CLASSIC.initial_board(), 1)
best_cell = max(prior, key=prior.get)
print(f"trained net's opening prior favors cell {best_cell} "
      f"({eng.cell_type(best_cell)}) at {prior[best_cell]:.2f}")

trained net's opening prior favors cell 5 (edge) at 0.74


Eight games is nowhere near enough to train a strong network — this is a mechanism demo, not a strength demo — but the loss curve's direction and where the opening prior lands are both real outputs of real gradient steps, not staged.

## Configure the live search

Both play-against-MCTS boards below share one `PlayConfig`. Adjust it here, then re-run
the two cells that build the boards to feel the difference: `mode`/`c` change selection
(same PUCT from above), `tree_reuse` decides whether the search keeps its work between
moves (via the same `reroot` used in the tree-reuse section), and `computer_starts`
decides who moves first. By default the rollout itself is heuristic-guided (win > block >
center/corner/edge, see `mcts.make_heuristic_rollout_eval`) rather than pure random --
far less noisy on a board this small, and it's why the search below can play solidly at
modest iteration counts.

A `prior_style="lines"` option -- bias priors toward cells that complete or block a
winning line -- is a natural next step but isn't wired up yet; only `"uniform"` and
`"band"` (the center/corner/edge weights used throughout this notebook) work today.

**What to notice while you play:** every node still accumulates `N` (visits), `W` (total
value), `Q = W/N`; every iteration is still one traversal from the root to a leaf; and
toggling `tree_reuse` on vs. off is the same reused-subtree-vs-fresh-search contrast from
the tree reuse section above, now happening move by move instead of in one isolated
comparison.

In [13]:
play_cfg = sc.PlayConfig()  # mode="uct", c=1.414, n_iterations=500, heuristic_rollout, tree_reuse=True, computer_starts=True

import ipywidgets as W
from IPython.display import display, clear_output

iter_options = [25, 50, 100, 200, 300, 500, 800, 1200, 2000, 3000, 5000]
iter_slider = W.SelectionSlider(
    options=iter_options,
    value=play_cfg.n_iterations if play_cfg.n_iterations in iter_options else 500,
    description="n_iterations")
mode_dd = W.Dropdown(options=["uct", "puct"], value=play_cfg.mode, description="mode")
c_slider = W.FloatSlider(value=play_cfg.c, min=0.2, max=3.0, step=0.1, description="c")
reuse_cb = W.Checkbox(value=play_cfg.tree_reuse, description="tree_reuse")
starts_cb = W.Checkbox(value=play_cfg.computer_starts, description="computer_starts")
summary_out = W.Output()

def render_summary():
    with summary_out:
        clear_output(wait=True)
        print(play_cfg)

def on_change(_):
    play_cfg.n_iterations = iter_slider.value
    play_cfg.mode = mode_dd.value
    play_cfg.c = c_slider.value
    play_cfg.tree_reuse = reuse_cb.value
    play_cfg.computer_starts = starts_cb.value
    render_summary()

for _w in (iter_slider, mode_dd, c_slider, reuse_cb, starts_cb):
    _w.observe(on_change, names="value")

display(W.VBox([iter_slider, mode_dd, c_slider, reuse_cb, starts_cb, summary_out]))
render_summary()

**Budgets for reliable play** (measured by playing many games against the one-ply
heuristic below, alternating who moves first): with the corrected search and the
heuristic-guided rollout above, **Classic** never loses and draws become the normal
outcome by roughly **~200 iterations for UCT** or **~50-100 iterations for PUCT** with
band priors -- both comfortably under the panel's 500-iteration default. **Numerical**
has a much larger branching factor (45 legal moves at the opening vs. Classic's 9), so
the same reliability needs substantially more search: at a few hundred iterations the
first player (odds) already wins more often than not, but occasional losses remain; by
around **~5,000 iterations** the first-player advantage held with zero losses across
repeated testing. The panel above defaults to 500 to keep Classic snappy -- raise
`n_iterations` before playing Numerical if you want it closer to full strength.

## Play Classic against MCTS

The same search machinery from above, now reacting to whatever you actually play -- driven
by the `play_cfg` panel: `computer_starts` decides who's X, `mode`/`c`/`n_iterations`
control the search, and `tree_reuse` decides whether it keeps its tree between moves. The
status line below the board still flags any immediate threat your move creates before the
computer responds.

In [14]:
def classic_mcts_factory():
    return sc.build_mcts_from_playconfig(play_cfg, eng.CLASSIC, seed=random.randint(0, 2**31 - 1))

viz.build_human_vs_mcts(
    eng.CLASSIC, classic_mcts_factory,
    computer_iterations=play_cfg.n_iterations,
    iterations_for_move=play_cfg.iterations_for_move if play_cfg.use_graded_budget else None,
    tree_reuse=play_cfg.tree_reuse, computer_starts=play_cfg.computer_starts,
)

## Play Numerical against MCTS — the climax

The identical search machinery plays a completely different-feeling game once the win
condition becomes arithmetic instead of geometric -- still driven by the same `play_cfg`
panel above. Player 1 always owns the odds, player 2 the evens, and `computer_starts`
decides which one you are; click a cell, then pick which of your remaining numbers to
place there. The same threat scan now watches for lines that sum to 15.

In [15]:
def numerical_mcts_factory():
    return sc.build_mcts_from_playconfig(play_cfg, eng.NUMERICAL, seed=random.randint(0, 2**31 - 1))

viz.build_human_vs_mcts(
    eng.NUMERICAL, numerical_mcts_factory,
    computer_iterations=play_cfg.n_iterations,
    iterations_for_move=play_cfg.iterations_for_move if play_cfg.use_graded_budget else None,
    tree_reuse=play_cfg.tree_reuse, computer_starts=play_cfg.computer_starts,
)

## What you just saw

From minimax's exhaustive recursion, to the Monte Carlo method's sampling instead of enumerating, to MCTS spending that sampling budget where it matters most -- this notebook has been two games, one search: the same lazy-expansion tree, the same UCT/PUCT selection rule, the
same scalar negamax backprop, driving both a hand-analyzed opening and a live, configurable opponent.
Priors reshaped where the search spent its budget; c controlled how much it trusted them;
rerooting carried earlier work forward instead of discarding it; and a network trained
purely on its own self-play started to develop an opening preference of its own.

**Open question:** Numerical tic-tac-toe has a known forced win for the first player. Did
the network's self-play prior on the opening move end up pointing toward it — or somewhere
else entirely?